# Exercise 4 — Direct density estimation with normalizing flows

In the previous exercises we estimated density ratios directly with classifiers. Here we train two normalizing flows,

$$
\hat p_\mathrm{sig}(x), \qquad \hat p_\mathrm{bkg}(x),
$$

where $x=(x_1,\ldots,x_5)$ are the reconstructed/smeared features. We then:

1. train one binary-mask RealNVP flow for the background and one for the signal;
2. validate the learned densities against MC projections and the analytic smeared Gaussian-mixture truth;
3. inspect the requested **flow-density histograms** for $\hat p_S(x)$ on signal and $\hat p_B(x)$ on background;
4. build the likelihood directly from the densities;
5. use the **flow-based** estimates of $p_S$ and $p_B$ to throw pseudo-experiments and plot the expected distribution of the likelihood-ratio test statistic.

The main pedagogical contrast is:

- classifier method: learn $p_i(x)/p_\mathrm{ref}(x)$ directly;
- flow method: learn $p_i(x)$ directly and insert those densities in the likelihood.


## Google Colab setup

Run this first in Colab. It clones the tutorial helper code, installs the small set of extra dependencies, generates the toy samples if needed, and changes into the tutorial working directory so the relative paths below resolve correctly.


In [ ]:
## ============================================================================
# Google Colab setup — run me first. Safe to re-run; a no-op off Colab.
# ============================================================================
import os, sys

# --- config -----------------------------------------------------------------
REPO_URL = "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git"  # package + tutorial helpers
BRANCH = "ml4hep_school_tutorial"
N_BKG, N_SIG = 2_000_000, 2_000_000  # Colab-sized dataset; lower for a quicker pass
USE_DRIVE = True  # True -> save data/models to Google Drive so they persist across sessions
# ----------------------------------------------------------------------------

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        ROOT = "/content/drive/MyDrive/Colab Notebooks/ml4hep_tifr_colab"
    else:
        ROOT = "/content"
    os.makedirs(ROOT, exist_ok=True)
    os.chdir(ROOT)

    # 1) fetch ONLY the package source + tutorial helpers (skip Git-LFS / big blobs)
    if not os.path.isdir("nsbi-lhc-toolkit"):
        os.environ["GIT_LFS_SKIP_SMUDGE"] = "1"
        !git clone --depth 1 --filter=blob:none --sparse --branch $BRANCH $REPO_URL
        !cd nsbi-lhc-toolkit && git sparse-checkout set src workshops/ml4hep_tifr

    # 2) make `import nsbi_common_utils` work (pure-python src layout, no build step)
    src = os.path.abspath("nsbi-lhc-toolkit/src")
    if src not in sys.path:
        sys.path.insert(0, src)

    # 3) runtime deps Colab doesn't already ship
    !pip install -q pytorch-lightning onnx onnxruntime onnxscript iminuit mplhep

    # 4) work from the tutorial dir so utils.py / generate_distributions.py and the
    # ./dataframes, ./models_* relative paths resolve just like a local run
    os.chdir("nsbi-lhc-toolkit/workshops/ml4hep_tifr")

    # 5) generate the Gaussian-mixture samples if they aren't there yet
    if not os.path.exists("dataframes/signal.parquet"):
        !python generate_distributions.py --n_bkg $N_BKG --n_sig $N_SIG

print("Working dir:", os.getcwd())

## Inputs expected by this notebook

This notebook expects the updated generator output:

```text
dataframes/background.parquet
dataframes/signal.parquet
```

with both truth-level columns `z1,...,z5` and reco-level columns `x1,...,x5`. The fit below uses only the reco-level `x*` variables.


In [ ]:
import math
import os
from dataclasses import dataclass
from pathlib import Path

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import logsumexp
from scipy.stats import multivariate_normal

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import nsbi_common_utils

from utils import (
    FEATURES,
    background_components,
    signal_components,
    smearing_parameters,
    split_train_inference,
)

FEATURES = list(FEATURES)
N_DIM = len(FEATURES)

SEED = 12345
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Features: {FEATURES}")

In [ ]:
BASE_PATH = Path("./dataframes")
FLOW_MODEL_DIR = Path("models_flows")
DENSITY_DIR = Path("saved_densities_flows")
PLOT_DIR = Path("plots_flows")

for directory in [FLOW_MODEL_DIR, DENSITY_DIR, PLOT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Must match the density-ratio training/fitting notebooks if you want disjoint
# train/inference samples across all approaches.
SPLIT_SEED = 0
TRAIN_FRACTION = 0.5

# Keep these modest for a tutorial. Increase N_EPOCHS and MAX_TRAIN_EVENTS
# for tighter closure against the analytic truth.
MAX_TRAIN_EVENTS = {
    "background": 1_000_000,
    "signal": 1_000_000,
}

N_COUPLING_LAYERS = 8
HIDDEN_FEATURES = 1024
HIDDEN_LAYERS = 4
BATCH_SIZE = 2048
N_EPOCHS = 25
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0
VALIDATION_FRACTION = 0.20
PATIENCE = 5

# If checkpoints already exist, load them instead of retraining.
LOAD_IF_AVAILABLE = False

## Load data and make the same train/inference split

The flows are trained on the training half. The likelihood fit is evaluated on the inference half. This mirrors the density-ratio notebooks and avoids using the same events to learn and evaluate the likelihood.


In [ ]:
signal = pd.read_parquet(BASE_PATH / "signal.parquet")
background = pd.read_parquet(BASE_PATH / "background.parquet")

missing = [f for f in FEATURES if f not in signal.columns or f not in background.columns]
if missing:
    raise RuntimeError(
        "Missing reco-level columns "
        f"{missing}. Regenerate the parquet files with the updated generator "
        "that writes x1,...,x5."
    )

for name, df in [("signal", signal), ("background", background)]:
    if "weight" not in df.columns:
        raise RuntimeError(f"{name} dataframe is missing a 'weight' column.")
    print(
        f"{name:10s}: {len(df):,} events, "
        f"sum weights = {df['weight'].sum():.6g}"
    )

In [ ]:
signal_train, _ = split_train_inference(
    signal, train_fraction=TRAIN_FRACTION, seed=SPLIT_SEED
)
background_train, _ = split_train_inference(
    background, train_fraction=TRAIN_FRACTION, seed=SPLIT_SEED
)

TOTAL_YIELD = {
    "signal": float(signal_train["weight"].sum()),
    "background": float(background_train["weight"].sum()),
}

# Tutorial shortcut: use the same sample to train and evaluate.
# This keeps the exercise compact, but should not be done in a real analysis.
signal_eval = signal_train
background_eval = background_train

print(f"Flow training: {len(signal_train):,} signal + {len(background_train):,} background events")
print(f"Fit/eval: {len(signal_eval):,} signal + {len(background_eval):,} background events")
print(TOTAL_YIELD)

## A minimal RealNVP density estimator

This is a deliberately compact RealNVP implementation. Each affine-coupling layer uses a fixed binary mask, alternating which coordinates are transformed. We train by maximizing the exact log likelihood

$$
\log \hat p(x) = \log p_Z\big(f(x)\big) + \log\left|\det\frac{\partial f}{\partial x}\right|.
$$

For numerical stability, each sample is standardized before training. The final `flow_log_prob_x` helper below adds the standardization Jacobian back, so it returns a density in the original `x` coordinates.


In [ ]:
@dataclass
class Standardizer:
    mean: np.ndarray
    std: np.ndarray

    @classmethod
    def fit(cls, x):
        x = np.asarray(x, dtype=np.float32)
        mean = x.mean(axis=0).astype(np.float32)
        std = x.std(axis=0).astype(np.float32)
        std = np.where(std > 1e-6, std, 1.0).astype(np.float32)
        return cls(mean=mean, std=std)

    def transform(self, x):
        x = np.asarray(x, dtype=np.float32)
        return ((x - self.mean) / self.std).astype(np.float32)

    def inverse(self, z):
        z = np.asarray(z, dtype=np.float32)
        return (z * self.std + self.mean).astype(np.float32)

    @property
    def log_det_x_to_z_standardization(self):
        # z = (x - mean) / std, so log |dz/dx| = -sum(log std)
        return float(-np.log(self.std).sum())


class MLP(nn.Module):
    def __init__(self, n_in, n_out, hidden_features=128, hidden_layers=2):
        super().__init__()
        layers = []
        last = n_in
        for _ in range(hidden_layers):
            layers.extend([nn.Linear(last, hidden_features), nn.ReLU()])
            last = hidden_features
        layers.append(nn.Linear(last, n_out))
        self.net = nn.Sequential(*layers)

        # Start each coupling layer close to the identity map.
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, x):
        return self.net(x)


class AffineCoupling(nn.Module):
    def __init__(self, n_features, mask, hidden_features=128, hidden_layers=2, scale_clip=1.5):
        super().__init__()
        self.register_buffer("mask", torch.as_tensor(mask, dtype=torch.float32))
        self.net = MLP(
            n_features,
            2 * n_features,
            hidden_features=hidden_features,
            hidden_layers=hidden_layers,
        )
        self.scale_clip = float(scale_clip)

    def _shift_and_log_scale(self, x_masked):
        shift, log_scale = self.net(x_masked).chunk(2, dim=-1)
        inv_mask = 1.0 - self.mask
        log_scale = self.scale_clip * torch.tanh(log_scale) * inv_mask
        shift = shift * inv_mask
        return shift, log_scale

    def forward(self, x):
        """Map data space -> base space for this layer."""
        x_masked = x * self.mask
        shift, log_scale = self._shift_and_log_scale(x_masked)
        z = x_masked + (1.0 - self.mask) * (x - shift) * torch.exp(-log_scale)
        log_det = -log_scale.sum(dim=-1)
        return z, log_det

    def inverse(self, z):
        """Map base space -> data space for this layer."""
        z_masked = z * self.mask
        shift, log_scale = self._shift_and_log_scale(z_masked)
        x = z_masked + (1.0 - self.mask) * (z * torch.exp(log_scale) + shift)
        log_det = log_scale.sum(dim=-1)
        return x, log_det


class RealNVP(nn.Module):
    def __init__(
        self,
        n_features,
        n_coupling_layers=8,
        hidden_features=128,
        hidden_layers=2,
    ):
        super().__init__()
        base_mask = torch.tensor(
            [(i % 2) for i in range(n_features)], dtype=torch.float32
        )
        masks = [base_mask if i % 2 == 0 else 1.0 - base_mask for i in range(n_coupling_layers)]
        self.layers = nn.ModuleList(
            [
                AffineCoupling(
                    n_features=n_features,
                    mask=mask,
                    hidden_features=hidden_features,
                    hidden_layers=hidden_layers,
                )
                for mask in masks
            ]
        )
        self.n_features = int(n_features)

    def log_prob(self, x):
        z = x
        total_log_det = torch.zeros(x.shape[0], device=x.device)
        for layer in self.layers:
            z, log_det = layer(z)
            total_log_det = total_log_det + log_det
        base_log_prob = -0.5 * (z.pow(2) + math.log(2.0 * math.pi)).sum(dim=-1)
        return base_log_prob + total_log_det

    @torch.no_grad()
    def sample(self, n):
        z = torch.randn(n, self.n_features, device=next(self.parameters()).device)
        x = z
        for layer in reversed(self.layers):
            x, _ = layer.inverse(x)
        return x

## Training helpers

The `train_flow` helper saves checkpoints under `models_flows/`. Re-running the notebook will load existing checkpoints if `LOAD_IF_AVAILABLE = True`.


In [ ]:
def model_config():
    return {
        "n_features": N_DIM,
        "n_coupling_layers": N_COUPLING_LAYERS,
        "hidden_features": HIDDEN_FEATURES,
        "hidden_layers": HIDDEN_LAYERS,
    }


def build_flow():
    return RealNVP(**model_config()).to(device)


def checkpoint_path(sample_name):
    return FLOW_MODEL_DIR / f"realnvp_{sample_name}.pt"


def _torch_load(path):
    # PyTorch versions differ in whether torch.load exposes weights_only.
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


def save_flow(sample_name, flow, scaler):
    path = checkpoint_path(sample_name)
    torch.save(
        {
            "state_dict": flow.state_dict(),
            "scaler_mean": scaler.mean,
            "scaler_std": scaler.std,
            "features": FEATURES,
            "config": model_config(),
        },
        path,
    )
    return path


def load_flow(sample_name):
    path = checkpoint_path(sample_name)
    ckpt = _torch_load(path)
    flow = RealNVP(**ckpt["config"]).to(device)
    flow.load_state_dict(ckpt["state_dict"])
    flow.eval()
    scaler = Standardizer(
        mean=np.asarray(ckpt["scaler_mean"], dtype=np.float32),
        std=np.asarray(ckpt["scaler_std"], dtype=np.float32),
    )
    return {"flow": flow, "scaler": scaler, "path": path}


def choose_training_array(df, sample_name, seed):
    n_max = MAX_TRAIN_EVENTS.get(sample_name)
    if n_max is not None and len(df) > n_max:
        df = df.sample(n=n_max, random_state=seed).reset_index(drop=True)
    return df[FEATURES].to_numpy(dtype=np.float32)


def make_loaders(x_scaled, seed):
    x_tensor = torch.tensor(x_scaled, dtype=torch.float32)
    n_total = len(x_tensor)
    n_val = max(1, int(round(VALIDATION_FRACTION * n_total)))
    n_train = n_total - n_val

    generator = torch.Generator().manual_seed(seed)
    permutation = torch.randperm(n_total, generator=generator)
    train_idx = permutation[:n_train]
    val_idx = permutation[n_train:]

    train_ds = TensorDataset(x_tensor[train_idx])
    val_ds = TensorDataset(x_tensor[val_idx])

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False,
        generator=generator,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        drop_last=False,
    )
    return train_loader, val_loader


def train_flow(sample_name, df_train, seed=0):
    path = checkpoint_path(sample_name)
    if LOAD_IF_AVAILABLE and path.exists():
        print(f"Loading existing {sample_name} flow from {path}")
        return load_flow(sample_name)

    x = choose_training_array(df_train, sample_name=sample_name, seed=seed)
    scaler = Standardizer.fit(x)
    x_scaled = scaler.transform(x)
    train_loader, val_loader = make_loaders(x_scaled, seed=seed)

    flow = build_flow()
    optimizer = torch.optim.AdamW(
        flow.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )

    best_val = np.inf
    best_state = None
    stale_epochs = 0

    print(f"Training {sample_name} flow on {len(x_scaled):,} events")
    for epoch in range(1, N_EPOCHS + 1):
        flow.train()
        train_losses = []
        for (batch,) in train_loader:
            batch = batch.to(device)
            loss = -flow.log_prob(batch).mean()
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(flow.parameters(), max_norm=5.0)
            optimizer.step()
            train_losses.append(float(loss.detach().cpu()))

        flow.eval()
        val_losses = []
        with torch.no_grad():
            for (batch,) in val_loader:
                batch = batch.to(device)
                val_losses.append(float((-flow.log_prob(batch).mean()).detach().cpu()))
        train_loss = float(np.mean(train_losses))
        val_loss = float(np.mean(val_losses))
        print(f" epoch {epoch:03d}: train NLL = {train_loss:.4f}, val NLL = {val_loss:.4f}")

        if val_loss < best_val - 1e-4:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in flow.state_dict().items()}
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= PATIENCE:
                print(f" early stopping after {epoch} epochs")
                break

    if best_state is not None:
        flow.load_state_dict(best_state)
    flow.eval()
    saved = save_flow(sample_name, flow, scaler)
    print(f"Saved {sample_name} flow to {saved}")
    return {"flow": flow, "scaler": scaler, "path": saved}

In [ ]:
flows = {
    "background": train_flow("background", background_train, seed=101),
    "signal": train_flow("signal", signal_train, seed=202),
}

## Density and sampling helpers

`flow_log_prob_x` returns log densities in the original `x` coordinates, not standardized coordinates.


In [ ]:
@torch.no_grad()
def flow_log_prob_x(flow_pack, x, batch_size=65_536):
    if isinstance(x, pd.DataFrame):
        x = x[FEATURES].to_numpy(dtype=np.float32)
    else:
        x = np.asarray(x, dtype=np.float32)

    flow = flow_pack["flow"]
    scaler = flow_pack["scaler"]
    flow.eval()

    chunks = []
    for start in range(0, len(x), batch_size):
        xb = scaler.transform(x[start:start + batch_size])
        xb = torch.tensor(xb, dtype=torch.float32, device=device)
        log_p_scaled = flow.log_prob(xb).detach().cpu().numpy()
        log_p_x = log_p_scaled + scaler.log_det_x_to_z_standardization
        chunks.append(log_p_x)
    return np.concatenate(chunks)


@torch.no_grad()
def flow_sample_x(flow_pack, n, batch_size=65_536):
    flow = flow_pack["flow"]
    scaler = flow_pack["scaler"]
    flow.eval()
    chunks = []
    remaining = int(n)
    while remaining > 0:
        m = min(batch_size, remaining)
        z = flow.sample(m).detach().cpu().numpy()
        chunks.append(scaler.inverse(z))
        remaining -= m
    return np.concatenate(chunks, axis=0)

## Validation 1: MC projection closure

Generate samples from the trained flow and compare one-dimensional projections with the evaluation MC sample.


In [ ]:
N_FLOW_PROJECTION = 100_000
rng_validation = np.random.default_rng(SEED + 1)

fig, axes = plt.subplots(2, N_DIM, figsize=(4 * N_DIM, 6), sharey="row")
for row, (sample_name, df_eval) in enumerate(
    [("background", background_eval), ("signal", signal_eval)]
):
    n_mc = min(N_FLOW_PROJECTION, len(df_eval))
    df_plot = df_eval.sample(n=n_mc, random_state=SEED + row).reset_index(drop=True)
    x_mc = df_plot[FEATURES].to_numpy(dtype=np.float32)
    w_mc = df_plot["weight"].to_numpy(dtype=float)
    x_flow = flow_sample_x(flows[sample_name], n_mc)

    for col, feature in enumerate(FEATURES):
        ax = axes[row, col]
        lo, hi = np.percentile(np.concatenate([x_mc[:, col], x_flow[:, col]]), [0.5, 99.5])
        bins = np.linspace(lo, hi, 60)
        ax.hist(x_mc[:, col], bins=bins, weights=w_mc, density=True, histtype="step", label="MC")
        ax.hist(x_flow[:, col], bins=bins, density=True, histtype="step", linestyle="--", label="flow")
        ax.set_title(f"{sample_name}: {feature}")
        if col == 0:
            ax.set_ylabel("density")
        if row == 0 and col == N_DIM - 1:
            ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(PLOT_DIR / "flow_projection_closure.png", dpi=160, bbox_inches="tight")
plt.show()

## Analytic smeared Gaussian-mixture truth

The reco variables are generated as

$$
x = D y + \epsilon, \qquad \epsilon_i \sim \mathcal{N}(0, \sigma_{\mathrm{res},i}^2),
$$

where $D = \mathrm{diag}(\mathrm{scale})$. Therefore each Gaussian component remains Gaussian after smearing:

$$
\mu_x = D\mu_y, \qquad \Sigma_x = D\Sigma_yD^T + \mathrm{diag}(\sigma_\mathrm{res}^2).
$$

This analytic truth is only available because the tutorial toy model is simple.


In [ ]:
def smeared_components(components):
    """Map truth-level Gaussian mixture components to reco-level x components."""
    scale, resolution = smearing_parameters()
    scale = np.asarray(scale, dtype=float)
    resolution = np.asarray(resolution, dtype=float)
    D = np.diag(scale)
    out = []
    for frac, mean, cov in components:
        mean_x = scale * np.asarray(mean, dtype=float)
        cov_x = D @ np.asarray(cov, dtype=float) @ D.T + np.diag(resolution ** 2)
        out.append((float(frac), mean_x, cov_x))
    return out


TRUTH_COMPONENTS_X = {
    "background": smeared_components(background_components()),
    "signal": smeared_components(signal_components()),
}


def truth_log_prob_x(sample_name, x):
    """Analytic log p(x) for the smeared tutorial Gaussian mixture."""
    x = np.asarray(x, dtype=float)
    components = TRUTH_COMPONENTS_X[sample_name]
    fracs = np.asarray([c[0] for c in components], dtype=float)
    fracs = fracs / fracs.sum()
    terms = []
    for frac, (_, mean, cov) in zip(fracs, components):
        terms.append(np.log(frac) + multivariate_normal(mean=mean, cov=cov).logpdf(x))
    return logsumexp(np.vstack(terms), axis=0)


def sample_eval_array(df, n, seed):
    n = min(int(n), len(df))
    return df.sample(n=n, random_state=seed)[FEATURES].to_numpy(dtype=np.float32)

## Truth-based validation 1: direct log-density closure

Compare each flow's log density with the analytic smeared Gaussian-mixture truth on the corresponding sample.


In [ ]:
N_TRUTH_VALIDATION = 100_000
truth_validation_points = {
    "background": sample_eval_array(background_eval, N_TRUTH_VALIDATION, SEED + 10),
    "signal": sample_eval_array(signal_eval, N_TRUTH_VALIDATION, SEED + 11),
}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, sample_name in zip(axes, ["background", "signal"]):
    x = truth_validation_points[sample_name]
    log_p_flow = flow_log_prob_x(flows[sample_name], x)
    log_p_truth = truth_log_prob_x(sample_name, x)
    finite = np.isfinite(log_p_flow) & np.isfinite(log_p_truth)
    log_p_flow = log_p_flow[finite]
    log_p_truth = log_p_truth[finite]

    lo, hi = np.percentile(np.concatenate([log_p_truth, log_p_flow]), [0.5, 99.5])
    ax.hist2d(log_p_truth, log_p_flow, bins=80, range=[[lo, hi], [lo, hi]])
    ax.plot([lo, hi], [lo, hi], "k--", lw=1)
    ax.set_xlabel(r"truth $\log p(x)$")
    ax.set_ylabel(r"flow $\log \hat p(x)$")
    ax.set_title(sample_name)
fig.tight_layout()
fig.savefig(PLOT_DIR / "truth_log_density_closure.png", dpi=160, bbox_inches="tight")
plt.show()

## Truth-based validation 2: log-density-ratio closure

Compare the flow-based log density ratio with the analytic truth ratio. This is the second truth-based validation; the requested non-log density histograms are inserted immediately below.


In [ ]:
N_RATIO_VALIDATION = 100_000
ratio_validation_samples = {
    "background eval": sample_eval_array(background_eval, N_RATIO_VALIDATION, SEED + 20),
    "signal eval": sample_eval_array(signal_eval, N_RATIO_VALIDATION, SEED + 21),
}

fig, ax = plt.subplots(figsize=(7, 4.5))
ratio_residuals = {}
for label, x in ratio_validation_samples.items():
    log_r_flow = flow_log_prob_x(flows["signal"], x) - flow_log_prob_x(flows["background"], x)
    log_r_truth = truth_log_prob_x("signal", x) - truth_log_prob_x("background", x)
    residual = log_r_flow - log_r_truth
    residual = residual[np.isfinite(residual)]
    ratio_residuals[label] = residual
    ax.hist(residual, bins=80, density=True, histtype="step", label=label)

ax.axvline(0.0, color="k", lw=1, ls="--")
ax.set_xlabel(r"$\log(\hat p_S/\hat p_B) - \log(p_S/p_B)_\mathrm{truth}$")
ax.set_ylabel("density")
ax.set_title("Flow density-ratio closure against analytic truth")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(PLOT_DIR / "truth_log_ratio_closure.png", dpi=160, bbox_inches="tight")
plt.show()

## Requested addition: histograms of the flow densities, not their logs

The first panel is a simple histogram of $\hat p_S(x)$ evaluated on signal events. The second panel is the equivalent histogram of $\hat p_B(x)$ evaluated on background events.


In [ ]:
N_DENSITY_HIST = 100_000
x_signal_for_ps = sample_eval_array(signal_eval, N_DENSITY_HIST, SEED + 30)
x_background_for_pb = sample_eval_array(background_eval, N_DENSITY_HIST, SEED + 31)

p_s_flow_on_signal = np.exp(flow_log_prob_x(flows["signal"], x_signal_for_ps))
p_b_flow_on_background = np.exp(flow_log_prob_x(flows["background"], x_background_for_pb))

p_s_flow_on_signal = p_s_flow_on_signal[np.isfinite(p_s_flow_on_signal)]
p_b_flow_on_background = p_b_flow_on_background[np.isfinite(p_b_flow_on_background)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(p_s_flow_on_signal, bins=80, histtype="step")
axes[0].set_xlabel(r"flow-based $\hat p_S(x)$ on signal events")
axes[0].set_ylabel("events")
axes[0].set_title(r"Signal sample: $\hat p_S(x)$")

axes[1].hist(p_b_flow_on_background, bins=80, histtype="step")
axes[1].set_xlabel(r"flow-based $\hat p_B(x)$ on background events")
axes[1].set_ylabel("events")
axes[1].set_title(r"Background sample: $\hat p_B(x)$")

fig.tight_layout()
fig.savefig(PLOT_DIR / "requested_flow_density_histograms.png", dpi=160, bbox_inches="tight")
plt.show()

## Direct unbinned likelihood from the learned densities

For an event sample $\{x_i\}$, with fixed expected background yield $B$ and signal yield $S$ at $\mu=1$, the extended likelihood is

$$
\log L(\mu) = - (\mu S + B) + \sum_i \log\left[\mu S\,\hat p_S(x_i) + B\,\hat p_B(x_i)\right].
$$

For fits and toys it is numerically convenient to work with $r_i=\hat p_S(x_i)/\hat p_B(x_i)$ and log-likelihood differences relative to $\mu=0$:

$$
\Delta\log L(\mu) = -\mu S + \sum_i w_i\log\left(1 + \mu\,\frac{S}{B}r_i\right).
$$


In [ ]:
def safe_density_ratio(log_p_s, log_p_b, clip=50.0):
    """Return exp(log_p_s - log_p_b), clipped to avoid numerical overflow."""
    log_r = np.asarray(log_p_s) - np.asarray(log_p_b)
    return np.exp(np.clip(log_r, -clip, clip))


def delta_loglike_ratio(mu, r, weights, s_yield, b_yield):
    """log L(mu) - log L(0), dropping constants that cancel in likelihood ratios."""
    r = np.asarray(r, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)
    t = (s_yield / b_yield) * r
    return -mu * s_yield + np.sum(weights * np.log1p(mu * t))


def score_delta_loglike(mu, r, weights, s_yield, b_yield):
    """Derivative of delta_loglike_ratio with respect to mu."""
    r = np.asarray(r, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)
    t = (s_yield / b_yield) * r
    return -s_yield + np.sum(weights * t / (1.0 + mu * t))


def hessian_delta_loglike(mu, r, weights, s_yield, b_yield):
    """Second derivative of delta_loglike_ratio with respect to mu."""
    r = np.asarray(r, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)
    t = (s_yield / b_yield) * r
    return -np.sum(weights * (t ** 2) / (1.0 + mu * t) ** 2)


def fit_mu_from_density_ratio(
    r,
    weights,
    s_yield,
    b_yield,
    mu_min=0.0,
    mu_max=10.0,
    tol=1e-7,
    max_iter=60,
):
    """Constrained 1D MLE for mu using Newton steps bracketed by bisection."""
    r = np.asarray(r, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)

    score0 = score_delta_loglike(mu_min, r, weights, s_yield, b_yield)
    if score0 <= 0:
        return float(mu_min)

    lo = float(mu_min)
    hi = float(mu_max)
    while score_delta_loglike(hi, r, weights, s_yield, b_yield) > 0 and hi < 1e6:
        hi *= 2.0

    mu = 0.5 * (lo + hi)
    for _ in range(max_iter):
        score = score_delta_loglike(mu, r, weights, s_yield, b_yield)
        if abs(score) < tol * max(1.0, s_yield):
            break
        if score > 0:
            lo = mu
        else:
            hi = mu

        hess = hessian_delta_loglike(mu, r, weights, s_yield, b_yield)
        newton = mu - score / hess if hess < 0 else np.nan
        if not np.isfinite(newton) or newton <= lo or newton >= hi:
            newton = 0.5 * (lo + hi)
        mu = newton
    return float(mu)


def q0_from_density_ratio(r, weights, s_yield, b_yield):
    """Discovery test statistic q0 = -2 log lambda(0), profiling mu >= 0."""
    mu_hat = fit_mu_from_density_ratio(r, weights, s_yield, b_yield)
    q0 = 2.0 * max(0.0, delta_loglike_ratio(mu_hat, r, weights, s_yield, b_yield))
    return q0, mu_hat

## Asimov-style check on the evaluation sample

This uses weighted signal and background evaluation MC as an Asimov-like sample. The toy section below uses the same likelihood helpers, but generates pseudo-experiments from the flows themselves.


In [ ]:
MAX_FIT_EVENTS_PER_SAMPLE = None  # set to an int, e.g. 200_000, for a faster exploratory pass

def maybe_subsample(df, n_max, seed):
    if n_max is None or len(df) <= n_max:
        return df.reset_index(drop=True)
    return df.sample(n=n_max, random_state=seed).reset_index(drop=True)

bkg_fit = maybe_subsample(background_eval, MAX_FIT_EVENTS_PER_SAMPLE, SEED + 40)
sig_fit = maybe_subsample(signal_eval, MAX_FIT_EVENTS_PER_SAMPLE, SEED + 41)

x_fit = np.concatenate(
    [
        bkg_fit[FEATURES].to_numpy(dtype=np.float32),
        sig_fit[FEATURES].to_numpy(dtype=np.float32),
    ],
    axis=0,
)
w_fit = np.concatenate(
    [
        bkg_fit["weight"].to_numpy(dtype=np.float64),
        sig_fit["weight"].to_numpy(dtype=np.float64),
    ]
)

log_p_s_fit = flow_log_prob_x(flows["signal"], x_fit)
log_p_b_fit = flow_log_prob_x(flows["background"], x_fit)
r_fit = safe_density_ratio(log_p_s_fit, log_p_b_fit)

np.savez(
    DENSITY_DIR / "flow_density_ratio_fit_sample.npz",
    r_fit=r_fit,
    weights=w_fit,
    log_p_s_fit=log_p_s_fit,
    log_p_b_fit=log_p_b_fit,
    s_yield=TOTAL_YIELD["signal"],
    b_yield=TOTAL_YIELD["background"],
)

q0_asimov, mu_hat_asimov = q0_from_density_ratio(
    r_fit,
    w_fit,
    TOTAL_YIELD["signal"],
    TOTAL_YIELD["background"],
)
print(f"Asimov-like weighted fit: mu_hat = {mu_hat_asimov:.4f}, q0 = {q0_asimov:.4f}")

## Requested final addition: flow-based toys for the test-statistic distribution

The toys below are generated from the flow estimates themselves:

- background events are drawn from the background flow, $\hat p_B$;
- signal events are drawn from the signal flow, $\hat p_S$;
- each toy is fit with the direct density likelihood above;
- the plotted statistic is the discovery profile-likelihood statistic $q_0=-2\log\lambda(0)$.

For speed, the cell first draws reusable pools from the flows and then performs a Poisson bootstrap over those pools. In the large-pool limit this approaches ordinary unbinned pseudo-experiments while avoiding repeated density evaluation inside every toy.


In [ ]:
N_FLOW_TS_TOYS = 100
FLOW_TOY_POOL_SIZE = {
    "background": 50_000,
    "signal": 50_000,
}
FLOW_TS_TOY_LUMI_SCALE = 1.0  # scale both S and B expected yields; keep 1.0 for nominal toys
FLOW_TOY_SEED = SEED + 500

rng_toys = np.random.default_rng(FLOW_TOY_SEED)
S_TOY = FLOW_TS_TOY_LUMI_SCALE * TOTAL_YIELD["signal"]
B_TOY = FLOW_TS_TOY_LUMI_SCALE * TOTAL_YIELD["background"]

print(f"Toy expected yields: S = {S_TOY:.6g}, B = {B_TOY:.6g}")
print("Drawing reusable flow pools and evaluating p_S/p_B on them...")

def make_flow_ratio_pool(generator_name, n_pool):
    x_pool = flow_sample_x(flows[generator_name], int(n_pool))
    log_p_s = flow_log_prob_x(flows["signal"], x_pool)
    log_p_b = flow_log_prob_x(flows["background"], x_pool)
    return safe_density_ratio(log_p_s, log_p_b)

r_pool_background = make_flow_ratio_pool("background", FLOW_TOY_POOL_SIZE["background"])
r_pool_signal = make_flow_ratio_pool("signal", FLOW_TOY_POOL_SIZE["signal"])


def one_flow_q0_toy(mu_true, rng):
    """One pseudo-experiment generated from flow-based p_B and p_S."""
    # Poisson bootstrap weights represent repeated unbinned draws from each flow pool.
    w_background = rng.poisson(B_TOY / len(r_pool_background), size=len(r_pool_background)).astype(np.float64)
    if mu_true > 0:
        w_signal = rng.poisson(mu_true * S_TOY / len(r_pool_signal), size=len(r_pool_signal)).astype(np.float64)
    else:
        w_signal = np.zeros(len(r_pool_signal), dtype=np.float64)

    r_toy = np.concatenate([r_pool_background, r_pool_signal])
    w_toy = np.concatenate([w_background, w_signal])
    q0, mu_hat = q0_from_density_ratio(r_toy, w_toy, S_TOY, B_TOY)
    return q0, mu_hat, w_background.sum(), w_signal.sum()

print("Throwing background-only toys...")
toy_results_bonly = np.array(
    [one_flow_q0_toy(mu_true=0.0, rng=rng_toys) for _ in range(N_FLOW_TS_TOYS)]
)
print("Throwing signal-plus-background toys...")
toy_results_sb = np.array(
    [one_flow_q0_toy(mu_true=1.0, rng=rng_toys) for _ in range(N_FLOW_TS_TOYS)]
)

q0_bonly = toy_results_bonly[:, 0]
mu_hat_bonly = toy_results_bonly[:, 1]
q0_sb = toy_results_sb[:, 0]
mu_hat_sb = toy_results_sb[:, 1]

np.savez(
    DENSITY_DIR / "flow_based_q0_toys.npz",
    q0_bonly=q0_bonly,
    mu_hat_bonly=mu_hat_bonly,
    q0_sb=q0_sb,
    mu_hat_sb=mu_hat_sb,
    S_TOY=S_TOY,
    B_TOY=B_TOY,
    n_toys=N_FLOW_TS_TOYS,
    pool_background=FLOW_TOY_POOL_SIZE["background"],
    pool_signal=FLOW_TOY_POOL_SIZE["signal"],
)

upper = np.percentile(np.concatenate([q0_bonly, q0_sb]), 99.5)
upper = max(upper, 1.0)
bins = np.linspace(0.0, upper, 60)

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.hist(q0_bonly, bins=bins, density=True, histtype="step", label=r"$B$-only toys ($\mu_\mathrm{true}=0$)")
ax.hist(q0_sb, bins=bins, density=True, histtype="step", label=r"$S+B$ toys ($\mu_\mathrm{true}=1$)")
ax.set_xlabel(r"$q_0=-2\log\lambda(0)$")
ax.set_ylabel("toy density")
ax.set_title("Expected test-statistic distribution from flow-based toys")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(PLOT_DIR / "flow_based_toy_q0_distribution.png", dpi=160, bbox_inches="tight")
plt.show()

summary = pd.DataFrame(
    {
        "sample": ["B-only", "S+B"],
        "q0 median": [np.median(q0_bonly), np.median(q0_sb)],
        "q0 16%": [np.percentile(q0_bonly, 16), np.percentile(q0_sb, 16)],
        "q0 84%": [np.percentile(q0_bonly, 84), np.percentile(q0_sb, 84)],
        "mu_hat median": [np.median(mu_hat_bonly), np.median(mu_hat_sb)],
    }
)
summary